In [ ]:
import numpy as np

# -------------------------
# precision control
# -------------------------
# Switch between np.float64 and np.longdouble here but np.longdouble preferred bc it decreases the relative error of the finite-difference Hessian for smaller h's
DT = np.longdouble


def asDT(x):
    return np.asarray(x, dtype=DT)


# -------------------------
# constants
# -------------------------
R = DT("8.314")
T = DT("1000.0")  # K

# -------------------------
# Fake RKMP "database": binary Redlich-Kister coefficients
# L_ij(delta) = L0 + L1*delta + L2*delta^2 + ...
# Units: J/mol
# -------------------------
Lcoeff = {
    (0, 1): [DT("2000.0"), DT("-500.0"), DT("100.0")],
    (0, 2): [DT("1500.0"), DT("200.0")],
    (1, 2): [DT("3000.0"), DT("-1000.0"), DT("250.0")],
}

# -------------------------
# Helpers for L(delta), L'(delta), L''(delta)
# -------------------------
def L_poly(delta, coeffs):
    out = DT("0.0")
    p = DT("1.0")
    for c in coeffs:
        out += c * p
        p *= delta
    return out


def L_prime(delta, coeffs):
    out = DT("0.0")
    for v in range(1, len(coeffs)):
        out += v * coeffs[v] * (delta ** (v - 1))
    return out


def L_second(delta, coeffs):
    out = DT("0.0")
    for v in range(2, len(coeffs)):
        out += v * (v - 1) * coeffs[v] * (delta ** (v - 2))
    return out


# -------------------------
# Gibbs energy pieces (extensive, as function of moles n)
# -------------------------
def G_ref(n):
    # reference term is linear in n -> Hessian = 0
    # set to zero for verification
    return DT("0.0")


def G_id(n):
    # Ideal: G^id = RT * sum_i n_i ln(x_i), x_i = n_i / N
    n = asDT(n)
    if np.any(n <= 0):
        return DT(np.nan)
    N = np.sum(n, dtype=DT)
    x = n / N
    return R * T * np.sum(n * np.log(x), dtype=DT)


def G_ex(n):
    # Excess RKMP (pair-only) in extensive form:
    # G^ex = sum_{i<j} (n_i n_j / N) * L_ij((n_i - n_j) / N)
    n = asDT(n)
    if np.any(n <= 0):
        return DT(np.nan)
    N = np.sum(n, dtype=DT)

    G = DT("0.0")
    for (i, j), coeffs in Lcoeff.items():
        delta = (n[i] - n[j]) / N
        L = L_poly(delta, coeffs)
        G += (n[i] * n[j] / N) * L
    return G


def G_total(n):
    return G_ref(n) + G_id(n) + G_ex(n)


# -------------------------
# Analytic Hessian pieces with respect to moles
# -------------------------
def H_id(n):
    # d^2 G^id / dn_i dn_j = RT (delta_ij / n_i - 1 / N)
    n = asDT(n)
    N = np.sum(n, dtype=DT)
    C = len(n)
    H = np.zeros((C, C), dtype=DT)
    for i in range(C):
        for j in range(C):
            H[i, j] = R * T * ((DT("1.0") / n[i] if i == j else DT("0.0")) - DT("1.0") / N)
    return H


def H_ex(n):
    """
    Uses the pair-term formula
      Phi_ij = A * L(delta), with A = (n_i n_j) / N, delta = (n_i - n_j) / N
      Phi_kl = A_kl*L + A_k*L'*delta_l + A_l*L'*delta_k + A*L''*delta_k*delta_l + A*L'*delta_kl
    and sums over all pairs i < j.
    """
    n = asDT(n)
    N = np.sum(n, dtype=DT)
    C = len(n)

    def kron(a, b):
        return DT("1.0") if a == b else DT("0.0")

    H = np.zeros((C, C), dtype=DT)

    for (i, j), coeffs in Lcoeff.items():
        ni, nj = n[i], n[j]
        B = ni * nj
        D = ni - nj
        A = B / N
        delta = D / N

        L = L_poly(delta, coeffs)
        Lp = L_prime(delta, coeffs)
        Lpp = L_second(delta, coeffs)

        Bk = np.zeros(C, dtype=DT)
        Dk = np.zeros(C, dtype=DT)
        Ak = np.zeros(C, dtype=DT)
        dk = np.zeros(C, dtype=DT)

        for k in range(C):
            Bk[k] = kron(i, k) * nj + kron(j, k) * ni
            Dk[k] = kron(i, k) - kron(j, k)
            Ak[k] = Bk[k] / N - B / (N ** 2)
            dk[k] = Dk[k] / N - D / (N ** 2)

        for k in range(C):
            for ell in range(C):
                Bkell = kron(i, k) * kron(j, ell) + kron(j, k) * kron(i, ell)

                A_kell = (
                    Bkell / N
                    - Bk[k] / (N ** 2)
                    - Bk[ell] / (N ** 2)
                    + DT("2.0") * B / (N ** 3)
                )

                d_kell = (
                    -Dk[k] / (N ** 2)
                    - Dk[ell] / (N ** 2)
                    + DT("2.0") * D / (N ** 3)
                )

                H[k, ell] += (
                    A_kell * L
                    + Ak[k] * Lp * dk[ell]
                    + Ak[ell] * Lp * dk[k]
                    + A * Lpp * dk[k] * dk[ell]
                    + A * Lp * d_kell
                )

    return H


def H_analytic(n, include_ideal=True):
    H = H_ex(n)
    if include_ideal:
        H = H + H_id(n)
    return H


# -------------------------
# Finite-difference Hessian of G(n) with respect to moles
# -------------------------
def FD_H(n0, h):
    n0 = asDT(n0)
    h = DT(h)
    C = len(n0)

    if np.any(n0 - h <= 0):
        raise ValueError("Step h too large: n0 - h must stay > 0 for all components.")

    Hfd = np.zeros((C, C), dtype=DT)
    G0 = G_total(n0)

    for k in range(C):
        ek = np.zeros(C, dtype=DT)
        ek[k] = DT("1.0")
        Gp = G_total(n0 + h * ek)
        Gm = G_total(n0 - h * ek)
        Hfd[k, k] = (Gp - DT("2.0") * G0 + Gm) / (h ** 2)

    for k in range(C):
        for ell in range(k + 1, C):
            ek = np.zeros(C, dtype=DT)
            el = np.zeros(C, dtype=DT)
            ek[k] = DT("1.0")
            el[ell] = DT("1.0")

            Gpp = G_total(n0 + h * ek + h * el)
            Gpm = G_total(n0 + h * ek - h * el)
            Gmp = G_total(n0 - h * ek + h * el)
            Gmm = G_total(n0 - h * ek - h * el)

            val = (Gpp - Gpm - Gmp + Gmm) / (DT("4.0") * h ** 2)
            Hfd[k, ell] = val
            Hfd[ell, k] = val

    return Hfd


def fro_norm(M):
    M = asDT(M)
    return np.sqrt(np.sum(M * M, dtype=DT), dtype=DT)


# -------------------------
# RUN THE TEST
# -------------------------
n0 = np.array([DT("0.40"), DT("0.35"), DT("0.25")], dtype=DT)
Ha = H_analytic(n0, include_ideal=True)

print("Base G(n0) =", G_total(n0))

for h in [DT("1e-2"), DT("1e-3"), DT("1e-4"), DT("1e-5"), DT("1e-6")]:
    Hfd = FD_H(n0, h)
    denom = max(DT("1.0"), fro_norm(Ha))
    rel_err = fro_norm(Hfd - Ha) / denom
    print(f"h = {float(h):g}, relative error = {float(rel_err):.3e}")


Base G(n0) = -8300.002937587084942
h = 0.01, relative error = 2.693e-04
h = 0.001, relative error = 2.692e-06
h = 0.0001, relative error = 2.692e-08
h = 1e-05, relative error = 8.040e-10
h = 1e-06, relative error = 6.753e-08
